# COMP5318 Assignment 1: Rice Classification

##### Group number: 197
###### Student 1 SID: 530839244
###### Student 2 SID: 540958494
###### Student 3 SID: 550120560
###### Student 4 SID: ... 

## **1. Data Pre-processing**

In [317]:
# Import all libraries
import pandas as pd
import numpy as np

# Preprocessing 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier


# Analysis
from sklearn.model_selection import cross_val_score

In [318]:
# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [319]:
# Load the rice dataset: rice-final2.csv
rice_df = pd.read_csv("rice-final2.csv")

print(rice_df.head())


    Area    Perimiter Major_Axis_Length Minor_Axis_Length Eccentricity  \
0  12573  461.4660034       192.9033508       84.57207489  0.898771763   
1  12845  464.1210022       194.3322144       85.52433777  0.897951961   
2  14055  488.7489929       207.7517548       87.25032806  0.907536149   
3  14412  490.3240051       207.4761353       89.68951416  0.901735425   
4  14658  477.1170044       189.5666351       99.99777985  0.849550545   

  Convex_Area       Extent   class  
0       12893  0.550433397  class2  
1       13125  0.774962306  class2  
2       14484  0.550076306  class1  
3       14703  0.598853171  class1  
4       15048  0.649503708  class2  


In [320]:
##### Pre-process dataset #####

## Fill in missing attribute values ##

rice_df = rice_df.replace(['?', 'NA', 'N/A', 'na', ''], np.nan)        # Replace common blanks with NaN

x = rice_df.drop(columns=["class"])
y = rice_df["class"]

imp_mean = SimpleImputer(missing_values = np.nan, strategy = 'mean')  # Create imputer 
x = imp_mean.fit_transform(x)       # Find means of columns and replace blank values


## Normalise the data ##
scaler = MinMaxScaler()     # Create scaler
x = scaler.fit_transform(x)  # Normalise numerical data between 0 and 1

## Change class values ##
y.replace("class1", 0, inplace=True)
y.replace("class2", 1, inplace=True)

print(rice_df.head())


    Area    Perimiter Major_Axis_Length Minor_Axis_Length Eccentricity  \
0  12573  461.4660034       192.9033508       84.57207489  0.898771763   
1  12845  464.1210022       194.3322144       85.52433777  0.897951961   
2  14055  488.7489929       207.7517548       87.25032806  0.907536149   
3  14412  490.3240051       207.4761353       89.68951416  0.901735425   
4  14658  477.1170044       189.5666351       99.99777985  0.849550545   

  Convex_Area       Extent  class  
0       12893  0.550433397      1  
1       13125  0.774962306      1  
2       14484  0.550076306      0  
3       14703  0.598853171      0  
4       15048  0.649503708      1  


In [321]:
# Print first ten rows of pre-processed dataset to 4 decimal places as per assignment spec
# A function is provided to assist

def print_data(X, y, n_rows=10):
    """Takes a numpy data arraCy and target and prints the first ten rows.
    
    Arguments:
        X: numpy array of shape (n_examples, n_features)
        y: numpy array of shape (n_examples)
        n_rows: numpy of rows to print
    """
    for example_num in range(n_rows):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")

        if example_num == len(X)-1:
            print(y[example_num],end="")
        else:
            print(y[example_num])


print_data(x, y)
            


0.4628,0.5406,0.5113,0.4803,0.7380,0.4699,0.1196,1
0.4900,0.5547,0.5266,0.5018,0.7319,0.4926,0.8030,1
0.6109,0.6847,0.6707,0.5409,0.8032,0.6253,0.1185,0
0.6466,0.6930,0.6677,0.5961,0.7601,0.6467,0.2669,0
0.6712,0.6233,0.4755,0.8293,0.3721,0.6803,0.4211,1
0.2634,0.2932,0.2414,0.4127,0.5521,0.2752,0.2825,1
0.8175,0.9501,0.9515,0.5925,0.9245,0.8162,0.0000,0
0.3174,0.3588,0.3601,0.3908,0.6921,0.3261,0.8510,1
0.3130,0.3050,0.2150,0.5189,0.3974,0.3159,0.4570,1
0.5120,0.5237,0.4409,0.6235,0.5460,0.5111,0.3155,1


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning

In [322]:
## Setting the 10 fold stratified cross-validation
cvKFold=StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# The stratified folds from cvKFold should be provided to the classifiers

In [323]:
# Logistic Regression

log_reg_model = LogisticRegression().fit(x, y) # Create and fit Model

scores = cross_val_score(log_reg_model, x, y, cv=cvKFold) # Calculate scores for each fold 

print("-- Logistic Regression --")
print(f"Scores for each fold:")
for score in scores:
    print(f"{score:.3f}")
print(f"\nAverage score: {scores.mean():.3f}")
lr_score = scores.mean()


-- Logistic Regression --
Scores for each fold:
0.914
0.936
0.964
0.943
0.950
0.929
0.943
0.950
0.893
0.964

Average score: 0.939


In [324]:
# Naïve Bayes
nb_model = GaussianNB().fit(x, y)

scores = cross_val_score(nb_model, x, y, cv=cvKFold) # Calculate scores for each fold 

print("-- Gaussian Naives Bayes --")
print(f"Scores for each fold:")
for score in scores:
    print(f"{score:.3f}")
print(f"\nAverage score: {scores.mean():.3f}")
nb_score = scores.mean()


-- Gaussian Naives Bayes --
Scores for each fold:
0.900
0.929
0.957
0.914
0.943
0.936
0.943
0.943
0.864
0.936

Average score: 0.926


### Part 1 Results


In [325]:
# Print results for each classifier in part 1 to 4 decimal places here:
print(f"LogR average cross-validation accuracy: {lr_score:.4f}")
print(f"NB average cross-validation accuracy: {nb_score:.4f}")

LogR average cross-validation accuracy: 0.9386
NB average cross-validation accuracy: 0.9264


### Part 2: Cross-validation with parameter tuning

In [326]:
# KNN 
# parameters you may consider


k_values = [1, 3, 5, 7]
p_values = [1, 2]

max_score = 0
best_k = 0
best_p = 0

print("-- Gaussian Naives Bayes --")
print("Average score for each combination")

for k in k_values:
    for p in p_values:

        knn_model = KNeighborsClassifier(n_neighbors = k, p = p).fit(x, y)
        scores = cross_val_score(knn_model, x, y, cv=cvKFold)
        print(f"k = {k}, p = {p}, score = {scores.mean():.3f}")
        
        if scores.mean() > max_score:
            max_score = scores.mean()
            best_k = k
            best_p = p

# Print best score
print(f"\nBest Average score: {max_score:.3f}")
print(f"Best k: {k}")
print(f"Best p: {p}")
nb_score = scores.mean()



-- Gaussian Naives Bayes --
Average score for each combination
k = 1, p = 1, score = 0.906
k = 1, p = 2, score = 0.906
k = 3, p = 1, score = 0.930
k = 3, p = 2, score = 0.936
k = 5, p = 1, score = 0.934
k = 5, p = 2, score = 0.931
k = 7, p = 1, score = 0.937
k = 7, p = 2, score = 0.935

Best Average score: 0.937
Best k: 7
Best p: 2


In [327]:
# Decision Tree 
# parameters you may consider
max_depth = [3, 5, 7, 10]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]

max_score = 0
best_depth = 0
best_min_samples_split = 0
best_min_samples_leaf = 0

print("-- Decision Tree --")
print("Average score for each combination")

for depth in max_depth:
    for split in min_samples_split:
        for leaf in min_samples_leaf:

            dt_model = DecisionTreeClassifier(max_depth = depth, min_samples_split = split, min_samples_leaf = leaf).fit(x, y)
            scores = cross_val_score(dt_model, x, y, cv=cvKFold)
            print(f"max_depth = {depth},  min_samples_split = {split}, min_samples_leaf = {leaf}, score = {scores.mean():.3f}")
            
            if scores.mean() > max_score:
                max_score = scores.mean()
                best_depth = depth
                best_min_samples_split = split
                best_min_samples_leaf = leaf


# Print best score
print(f"\nBest Average score: {max_score:.3f}")
print(f"Best depth: {best_depth}")
print(f"Best min_samples_split: {best_min_samples_split}")
print(f"Best min_samples_leaf: {best_min_samples_leaf}")
dt_score = scores.mean()

-- Decision Tree --
Average score for each combination
max_depth = 3,  min_samples_split = 2, min_samples_leaf = 1, score = 0.941
max_depth = 3,  min_samples_split = 2, min_samples_leaf = 2, score = 0.941
max_depth = 3,  min_samples_split = 2, min_samples_leaf = 4, score = 0.941
max_depth = 3,  min_samples_split = 5, min_samples_leaf = 1, score = 0.941
max_depth = 3,  min_samples_split = 5, min_samples_leaf = 2, score = 0.941
max_depth = 3,  min_samples_split = 5, min_samples_leaf = 4, score = 0.941
max_depth = 3,  min_samples_split = 10, min_samples_leaf = 1, score = 0.941
max_depth = 3,  min_samples_split = 10, min_samples_leaf = 2, score = 0.941
max_depth = 3,  min_samples_split = 10, min_samples_leaf = 4, score = 0.941
max_depth = 5,  min_samples_split = 2, min_samples_leaf = 1, score = 0.929
max_depth = 5,  min_samples_split = 2, min_samples_leaf = 2, score = 0.929
max_depth = 5,  min_samples_split = 2, min_samples_leaf = 4, score = 0.929
max_depth = 5,  min_samples_split = 5, min

In [328]:
# Ada Boost
# parameters you may consider
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

In [329]:
# Gradient Boost
# parameters you may consider
max_depth = [1, 3, 5, 7]
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

In [330]:
# Random Forest
# You should use RandomForestClassifier from sklearn.ensemble with information gain and max_features set to ‘sqrt’.
# parameters you may consider
n_estimators = [10, 30, 60, 100]
max_leaf_nodes = [6, 12]



In [331]:
# SVM
# parameters you may consider
C = [0.01, 0.1, 1, 5]
gamma = [0.01, 0.1, 1, 10]
# optional
kernel = []


### Part 2: Results

In [332]:
# Perform Grid Search with 10-fold stratified cross-validation (GridSearchCV in sklearn). 
# The stratified folds from cvKFold should be provided to GridSearchV

# This should include using train_test_split from sklearn.model_selection with stratification and random_state=0
# Print results for each classifier here. All the reported results should be printed to 4 decimal places except for the integers such as "k", "p", n_estimators" and "max_leaf_nodes".

# example printing:
print(f"KNN best k: {best_k}")
print(f"KNN best p: {best_p}")
print("KNN cross-validation accuracy: ")
print("KNN test set accuracy: ")

...

print("RF best n_estimators: ")
print("RF best max_leaf_nodes: ")
print("RF cross-validation accuracy: ")
print("RF test set accuracy: ")
print("RF test set macro average F1: ")
print("RF test set weighted average F1: ")

KNN best k: 
KNN best p: 
KNN cross-validation accuracy: 
KNN test set accuracy: 
RF best n_estimators: 
RF best max_leaf_nodes: 
RF cross-validation accuracy: 
RF test set accuracy: 
RF test set macro average F1: 
RF test set weighted average F1: 


### Test your code

In [333]:
#load the test dataset to test out your model 


## **3. Reflection and Discussion**



this is the discussion
